In [0]:
%run ./nb_adls_connection

hospital_id,hospital_name,city,bed_count
H001,Apollo Main Hospital,Chennai,850
H002,Fortis Healthcare Delhi,Delhi,600
H003,Kokilaben Dhirubhai Hospital,Mumbai,700
H004,Manipal Hospital,Bangalore,400
H005,Yashoda Hospital,Hyderabad,450


patient_id,first_name,last_name,gender,dob,city
P001,Aarav,Sharma,M,1980-02-14,Mumbai
P002,Riya,Verma,F,1975-07-22,Delhi
P003,Kabir,Menon,M,1990-11-02,Bangalore
P004,Sneha,Rao,F,1988-09-19,Hyderabad
P005,Vikram,Singh,M,1965-05-05,Chennai


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

DataFrame[]

In [0]:
# 1. drop rows with no key
df_h = df_Hospital.na.drop(subset=["hospital_id"])
display(df_h)

hospital_id,hospital_name,city,bed_count
H001,Apollo Main Hospital,Chennai,850
H002,Fortis Healthcare Delhi,Delhi,600
H003,Kokilaben Dhirubhai Hospital,Mumbai,700
H004,Manipal Hospital,Bangalore,400
H005,Yashoda Hospital,Hyderabad,450


In [0]:
# 2. clean text + fix types

from pyspark.sql.functions import trim, expr
df_h = (df_h.withColumn("hospital_id", trim("hospital_id"))
            .withColumn("hospital_name", trim("hospital_name"))
            .withColumn("city", trim("city"))
            .withColumn("bed_count", expr("try_cast(bed_count as INT)")))


hospital_id,hospital_name,city,bed_count
H001,Apollo Main Hospital,Chennai,850
H002,Fortis Healthcare Delhi,Delhi,600
H003,Kokilaben Dhirubhai Hospital,Mumbai,700
H004,Manipal Hospital,Bangalore,400
H005,Yashoda Hospital,Hyderabad,450


In [0]:
# 3. fill remaining nulls
df_h = df_h.fillna({"city": "Unknown", "bed_count": 0})
display(df_h)


hospital_id,hospital_name,city,bed_count
H001,Apollo Main Hospital,Chennai,850
H002,Fortis Healthcare Delhi,Delhi,600
H003,Kokilaben Dhirubhai Hospital,Mumbai,700
H004,Manipal Hospital,Bangalore,400
H005,Yashoda Hospital,Hyderabad,450


In [0]:
# 4. remove duplicates
print(df_h.count())                                
df_h = df_h.dropDuplicates(["hospital_id"])
print(df_h.count())                                

5
5


In [0]:
from pyspark.sql.functions import current_timestamp
# renaming col
df_h = df_h.withColumnRenamed("bed_count", "total_beds")

#  add timestamp col
df_h = df_h.withColumn("load_ts", current_timestamp())
#  save as Delta table (Silver)
df_h.write.format("delta").mode("overwrite").saveAsTable("silver.hospital")
display(df_h)

hospital_id,hospital_name,city,total_beds,load_ts
H002,Fortis Healthcare Delhi,Delhi,600,2026-10-29T00:04:20.752Z
H004,Manipal Hospital,Bangalore,400,2026-10-29T00:04:20.752Z
H005,Yashoda Hospital,Hyderabad,450,2026-10-29T00:04:20.752Z
H001,Apollo Main Hospital,Chennai,850,2026-10-29T00:04:20.752Z
H003,Kokilaben Dhirubhai Hospital,Mumbai,700,2026-10-29T00:04:20.752Z


patient cleaning

In [0]:
#  nulls on key
df_p = df_Patient.na.drop(subset=["patient_id"])
display(df_p)

patient_id,first_name,last_name,gender,dob,city
P001,Aarav,Sharma,M,1980-02-14,Mumbai
P002,Riya,Verma,F,1975-07-22,Delhi
P003,Kabir,Menon,M,1990-11-02,Bangalore
P004,Sneha,Rao,F,1988-09-19,Hyderabad
P005,Vikram,Singh,M,1965-05-05,Chennai


In [0]:
# trim text + validate date

from pyspark.sql.functions import trim, col, coalesce, to_date, current_timestamp
df_p = (df_p.withColumn("patient_id", trim(col("patient_id")))
            .withColumn("first_name", trim(col("first_name")))
            .withColumn("last_name", trim(col("last_name")))
            .withColumn("dob", coalesce(to_date(col("dob"), "yyyy-MM-dd"),
                                        to_date(col("dob"), "dd/MM/yyyy"))))
display(df_p)

patient_id,first_name,last_name,gender,dob,city
P001,Aarav,Sharma,M,1980-02-14,Mumbai
P002,Riya,Verma,F,1975-07-22,Delhi
P003,Kabir,Menon,M,1990-11-02,Bangalore
P004,Sneha,Rao,F,1988-09-19,Hyderabad
P005,Vikram,Singh,M,1965-05-05,Chennai


In [0]:
# fill nulls and  duplicates
df_p = df_p.fillna({"city": "Unknown", "gender": "Unknown"})

df_p = df_p.dropDuplicates(["patient_id"])
display(df_p)

patient_id,first_name,last_name,gender,dob,city
P004,Sneha,Rao,F,1988-09-19,Hyderabad
P003,Kabir,Menon,M,1990-11-02,Bangalore
P005,Vikram,Singh,M,1965-05-05,Chennai
P001,Aarav,Sharma,M,1980-02-14,Mumbai
P002,Riya,Verma,F,1975-07-22,Delhi


In [0]:
# 5. rename and timestamp(adding new col)
df_p = (df_p.withColumnRenamed("dob", "date_of_birth")
            .withColumn("load_ts", current_timestamp()))
display(df_p)

patient_id,first_name,last_name,gender,date_of_birth,city,load_ts
P004,Sneha,Rao,F,1988-09-19,Hyderabad,2026-10-29T00:14:32.880Z
P003,Kabir,Menon,M,1990-11-02,Bangalore,2026-10-29T00:14:32.880Z
P005,Vikram,Singh,M,1965-05-05,Chennai,2026-10-29T00:14:32.880Z
P001,Aarav,Sharma,M,1980-02-14,Mumbai,2026-10-29T00:14:32.880Z
P002,Riya,Verma,F,1975-07-22,Delhi,2026-10-29T00:14:32.880Z


In [0]:
#  save in delta format
df_p.write.format("delta").mode("overwrite").saveAsTable("silver.patient")
display(df_p)

patient_id,first_name,last_name,gender,date_of_birth,city,load_ts
P004,Sneha,Rao,F,1988-09-19,Hyderabad,2026-10-29T00:16:18.451Z
P003,Kabir,Menon,M,1990-11-02,Bangalore,2026-10-29T00:16:18.451Z
P005,Vikram,Singh,M,1965-05-05,Chennai,2026-10-29T00:16:18.451Z
P001,Aarav,Sharma,M,1980-02-14,Mumbai,2026-10-29T00:16:18.451Z
P002,Riya,Verma,F,1975-07-22,Delhi,2026-10-29T00:16:18.451Z
